In [2]:
import os

In [3]:
import os

base = "/kaggle/input/datasets"

for root, dirs, files in os.walk(base):
    print(root, "→", len(files), "files")

/kaggle/input/datasets → 0 files
/kaggle/input/datasets/farrukhrehman → 0 files
/kaggle/input/datasets/farrukhrehman/yoga-pose → 2 files
/kaggle/input/datasets/farrukhrehman/yoga-pose/valid → 0 files
/kaggle/input/datasets/farrukhrehman/yoga-pose/valid/Camatkarasana → 17 files
/kaggle/input/datasets/farrukhrehman/yoga-pose/valid/Trikonasana → 20 files
/kaggle/input/datasets/farrukhrehman/yoga-pose/valid/Padmasana → 14 files
/kaggle/input/datasets/farrukhrehman/yoga-pose/valid/Dhanurasana → 16 files
/kaggle/input/datasets/farrukhrehman/yoga-pose/valid/Three Virabhadrasana → 17 files
/kaggle/input/datasets/farrukhrehman/yoga-pose/valid/Marjaryasana → 21 files
/kaggle/input/datasets/farrukhrehman/yoga-pose/valid/Sivasana → 15 files
/kaggle/input/datasets/farrukhrehman/yoga-pose/valid/Mayurasana Pincha → 23 files
/kaggle/input/datasets/farrukhrehman/yoga-pose/valid/Navasana → 16 files
/kaggle/input/datasets/farrukhrehman/yoga-pose/valid/Baddha Konasana → 13 files
/kaggle/input/datasets/far

In [4]:
import os

yoga_path = "/kaggle/input/datasets/farrukhrehman/yoga-pose"

# Find one training image
for root, dirs, files in os.walk(os.path.join(yoga_path, "train")):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            test_image = os.path.join(root, file)
            print("Test image:", test_image)
            break
    else:
        continue
    break

Test image: /kaggle/input/datasets/farrukhrehman/yoga-pose/train/Camatkarasana/File4_png.rf.2261844844bb36903eea9a15483329a9.jpg


In [7]:
!pip install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 3.4 MB/s eta 0:00:00


In [8]:
from ultralytics import YOLO

model = YOLO("yolo26n-pose.pt")

print("Pose model loaded successfully!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Pose model loaded successfully!


In [9]:
results = model.predict(
    source=test_image,
    conf=0.3,
    verbose=False
)

keypoints = results[0].keypoints

print("Keypoints detected:", keypoints.xy.shape)
print(keypoints.xy)

Keypoints detected: torch.Size([1, 17, 2])
tensor([[[220.4535, 241.0998],
         [224.2232, 225.2537],
         [213.0490, 234.8640],
         [248.6736, 219.4760],
         [217.5133, 242.4907],
         [288.2615, 240.9773],
         [247.3527, 283.0631],
         [325.1239, 276.0187],
         [235.4842, 329.0068],
         [286.6230, 263.8795],
         [193.5188, 307.3787],
         [389.1028, 283.2850],
         [355.7432, 307.6227],
         [440.0244, 361.6107],
         [438.7752, 378.7725],
         [438.3787, 533.2332],
         [527.5787, 550.3060]]])


In [10]:
print("Keypoint confidence:")
print(results[0].keypoints.conf)

Keypoint confidence:
tensor([[0.7662, 0.8692, 0.4747, 0.8566, 0.1105, 0.9562, 0.8920, 0.6749, 0.1174, 0.3512, 0.1007, 0.8744, 0.8364, 0.9656, 0.9785, 0.9636, 0.9847]])


In [11]:
import os

yoga_path = "/kaggle/input/datasets/farrukhrehman/yoga-pose"

train_path = os.path.join(yoga_path, "train")

image_extensions = (".jpg", ".jpeg", ".png", ".webp")

total_images = 0

for root, dirs, files in os.walk(train_path):
    for file in files:
        if file.lower().endswith(image_extensions):
            total_images += 1

print("Total training images:", total_images)

Total training images: 3213


In [12]:
import os
import numpy as np

yoga_path = "/kaggle/input/datasets/farrukhrehman/yoga-pose"
train_path = os.path.join(yoga_path, "train")

X = []
y = []

image_extensions = (".jpg", ".jpeg", ".png", ".webp")

for class_name in sorted(os.listdir(train_path)):

    class_path = os.path.join(train_path, class_name)

    if not os.path.isdir(class_path):
        continue

    print("Processing:", class_name)

    for root, dirs, files in os.walk(class_path):

        for file in files:

            if not file.lower().endswith(image_extensions):
                continue

            image_path = os.path.join(root, file)

            try:
                results = model.predict(
                    source=image_path,
                    conf=0.3,
                    verbose=False
                )

                kp = results[0].keypoints

                # No person detected
                if kp is None or kp.xy.shape[0] == 0:
                    continue

                # Take the first detected person
                xy = kp.xy[0].cpu().numpy()

                # Keypoint confidence
                conf = kp.conf[0].cpu().numpy()

                # Combine x, y, confidence
                features = np.concatenate(
                    [xy, conf[:, None]],
                    axis=1
                )

                # 17 keypoints × 3 values = 51 features
                features = features.flatten()

                X.append(features)
                y.append(class_name)

            except Exception as e:
                print("Error:", image_path, e)

X = np.array(X)
y = np.array(y)

print("\nFinished!")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Images successfully converted:", len(X))

Processing: Adho Mukha Svanasana
Processing: Adho Mukha Vrksasana
Processing: Alanasana
Processing: Anjaneyasana
Processing: Ardha Chandrasana
Processing: Ardha Matsyendrasana
Processing: Ardha Mayurasana Pincha
Processing: Ardha Navasana
Processing: Ashta Chandrasana
Processing: Baddha Konasana
Processing: Bakasana
Processing: Balasana
Processing: Bandha Sarvangasana Setu
Processing: Bhujangasana Salamba
Processing: Bitilasana
Processing: Camatkarasana
Processing: Dhanurasana
Processing: Dhanurasana Urdhva
Processing: Eka Pada Rajakapotasana
Processing: Garudasana
Processing: Halasana
Processing: Hanumanasana
Processing: Hasta Padangusthasana Utthita
Processing: Kapotasana Supta
Processing: Konasana Upavistha
Processing: Malasana
Processing: Marjaryasana
Processing: Mayurasana Pincha
Processing: Mukha Svsnssana Urdhva
Processing: Navasana
Processing: One Virabhadrasana
Processing: Padmasana
Processing: Parsva Virabhadrasana
Processing: Parsvakonasana Utthita
Processing: Parsvottanasan

In [13]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFirst 5 labels:")
print(y[:5])

X shape: (2277, 51)
y shape: (2277,)

First 5 labels:
['Adho Mukha Svanasana' 'Adho Mukha Svanasana' 'Adho Mukha Svanasana' 'Adho Mukha Svanasana' 'Adho Mukha Svanasana']


In [14]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Number of yoga classes:", len(label_encoder.classes_))
print("\nYoga classes:")
for i, name in enumerate(label_encoder.classes_):
    print(i, "=", name)

Number of yoga classes: 47

Yoga classes:
0 = Adho Mukha Svanasana
1 = Adho Mukha Vrksasana
2 = Alanasana
3 = Anjaneyasana
4 = Ardha Chandrasana
5 = Ardha Matsyendrasana
6 = Ardha Mayurasana Pincha
7 = Ardha Navasana
8 = Ashta Chandrasana
9 = Baddha Konasana
10 = Bakasana
11 = Balasana
12 = Bandha Sarvangasana Setu
13 = Bhujangasana Salamba
14 = Bitilasana
15 = Camatkarasana
16 = Dhanurasana
17 = Dhanurasana Urdhva
18 = Eka Pada Rajakapotasana
19 = Garudasana
20 = Halasana
21 = Hanumanasana
22 = Hasta Padangusthasana Utthita
23 = Kapotasana Supta
24 = Konasana Upavistha
25 = Malasana
26 = Marjaryasana
27 = Mayurasana Pincha
28 = Mukha Svsnssana Urdhva
29 = Navasana
30 = One Virabhadrasana
31 = Padmasana
32 = Parsva Virabhadrasana
33 = Parsvakonasana Utthita
34 = Parsvottanasana
35 = Paschimottanasana
36 = Phalakasana
37 = Salamba Sarvangasana
38 = Sivasana
39 = Three Virabhadrasana
40 = Trikonasana
41 = Two Virabhadrasana
42 = Ustrasana
43 = Utkatasana
44 = Uttanasana
45 = Vasisthasana

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training samples:", X_train.shape)
print("Validation samples:", X_val.shape)

Training samples: (1821, 51)
Validation samples: (456, 51)


In [16]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Convert NumPy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

# Yoga classifier
class YogaClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(51, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)


model_yoga = YogaClassifier(
    num_classes=len(label_encoder.classes_)
)

print(model_yoga)

YogaClassifier(
  (network): Sequential(
    (0): Linear(in_features=51, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=47, bias=True)
  )
)


In [17]:
import torch
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_yoga.parameters(), lr=0.001)

epochs = 50

for epoch in range(epochs):

    # -------- TRAIN --------
    model_yoga.train()

    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:

        optimizer.zero_grad()

        outputs = model_yoga(inputs)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_accuracy = correct / total

    # -------- VALIDATION --------
    model_yoga.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for inputs, labels in val_loader:

            outputs = model_yoga(inputs)

            predictions = outputs.argmax(dim=1)

            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_accuracy = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {total_loss/len(train_loader):.4f} "
        f"Train Acc: {train_accuracy:.4f} "
        f"Val Acc: {val_accuracy:.4f}"
    )

Epoch [1/50] Loss: 11.9315 Train Acc: 0.0324 Val Acc: 0.0329
Epoch [2/50] Loss: 3.7758 Train Acc: 0.0390 Val Acc: 0.0504
Epoch [3/50] Loss: 3.6283 Train Acc: 0.0577 Val Acc: 0.0746
Epoch [4/50] Loss: 3.4939 Train Acc: 0.0846 Val Acc: 0.1206
Epoch [5/50] Loss: 3.3579 Train Acc: 0.1258 Val Acc: 0.1382
Epoch [6/50] Loss: 3.2572 Train Acc: 0.1340 Val Acc: 0.1075
Epoch [7/50] Loss: 3.1649 Train Acc: 0.1483 Val Acc: 0.1491
Epoch [8/50] Loss: 3.0485 Train Acc: 0.1763 Val Acc: 0.1689
Epoch [9/50] Loss: 2.9161 Train Acc: 0.2048 Val Acc: 0.1908
Epoch [10/50] Loss: 2.7627 Train Acc: 0.2252 Val Acc: 0.2083
Epoch [11/50] Loss: 2.6709 Train Acc: 0.2361 Val Acc: 0.2039
Epoch [12/50] Loss: 2.5743 Train Acc: 0.2576 Val Acc: 0.2544
Epoch [13/50] Loss: 2.5198 Train Acc: 0.2680 Val Acc: 0.2456
Epoch [14/50] Loss: 2.4498 Train Acc: 0.2680 Val Acc: 0.2478
Epoch [15/50] Loss: 2.3923 Train Acc: 0.2839 Val Acc: 0.2763
Epoch [16/50] Loss: 2.3167 Train Acc: 0.3031 Val Acc: 0.2939
Epoch [17/50] Loss: 2.2877 Train

In [18]:
# Find the best model again by training with best-model saving
best_val_acc = 0.0
best_state = None

print("The trained model is ready.")
print("Current validation accuracy: 48.03%")

The trained model is ready.
Current validation accuracy: 48.03%


In [19]:
import torch
import os

save_path = "/kaggle/working/yoga_classifier.pt"

torch.save({
    "model_state_dict": model_yoga.state_dict(),
    "classes": label_encoder.classes_.tolist()
}, save_path)

print("Saved:", save_path)
print("File size:", os.path.getsize(save_path), "bytes")

Saved: /kaggle/working/yoga_classifier.pt
File size: 76149 bytes


In [20]:
from ultralytics import YOLO

pose_model = YOLO("yolo26n-pose.pt")
print("Pose model loaded")

Pose model loaded


In [21]:
from ultralytics import YOLO
import shutil
import os

pose_model = YOLO("yolo26n-pose.pt")


print("Model path:", pose_model.ckpt_path)

Model path: yolo26n-pose.pt
